# Notebook 3 (Fase 2) — API REST con FastAPI + MongoDB + ngrok
### Ernesto Investing AI · iDeSo · UNMSM — Semana 13

**Objetivo:** Levantar una API que LEE de MongoDB (no recalcula) y expone endpoints REST via ngrok, cubriendo todos los modulos generados por los Notebooks 1, 2, 4, 5, 6 y 7.

**Prerequisitos:** `NGROK_AUTHTOKEN` en Secrets de Colab. Notebooks previos ejecutados (colecciones `precios_ohlcv`, `predicciones`, `metricas_modelos`, `resultados_backtest`, `predicciones_lstm`, `metricas_lstm` y `sentimiento_nlp` pobladas).

**Endpoints:**
- `GET /api/salud` — estado del servidor y de la base de datos (todas las colecciones)
- `GET /api/mercado/{ticker}` — datos OHLCV + indicadores tecnicos (Notebook 1)
- `GET /api/svc/{ticker}` — prediccion y metricas del clasificador SVC (Notebook 2)
- `GET /api/rnn/{ticker}` — predicciones y metricas de las 4 arquitecturas recurrentes: LSTM, BiLSTM, GRU, SimpleRNN (Notebook 4)
- `GET /api/rnn/{ticker}/{modelo}` — prediccion y metricas de una arquitectura RNN especifica
- `GET /api/backtest/{ticker}` — resultados del backtest (estrategia SMA 20/50), operaciones y curva de equity (Notebook 5)
- `GET /api/lstm-regressor/{ticker}` — pronostico de precios (7/14/30/60 dias) y metricas del LSTM Regressor vs. ARIMA (Notebook 6)
- `GET /api/sentimiento/{ticker}` — analisis de sentimiento NLP (VADER) de noticias financieras (Notebook 7)
- `GET /api/resumen/{ticker}` — resumen consolidado de TODOS los modulos para un ticker (uso en dashboard)

**Pipeline:** MongoDB (todas las colecciones) → FastAPI → ngrok → frontend

In [17]:
# Paso 1 — Instalar librerias necesarias
!pip install fastapi uvicorn pyngrok nest-asyncio "pymongo[srv]" --quiet

print("Librerias instaladas correctamente")

Librerias instaladas correctamente


In [18]:
# Paso 2 — Configurar credenciales: ngrok (Secrets) y MongoDB Atlas
# El authtoken de ngrok se lee de los Secrets de Colab, tal como pide la
# especificacion (nunca se escribe en texto plano en el notebook).
# La password de MongoDB se pide de forma oculta con getpass, siguiendo el
# mismo patron de conexion usado en los Notebooks 1, 2, 4, 5, 6 y 7.
from google.colab import userdata
from getpass import getpass
from pyngrok import conf
from pymongo import MongoClient
from pymongo.server_api import ServerApi

# --- ngrok: pyngrok 8.x usa conf.get_default().auth_token ---
conf.get_default().auth_token = userdata.get("NGROK_AUTHTOKEN")

# --- MongoDB Atlas ---
DB_USER = "gdev03_db_user"
DB_PASSWORD = getpass("Password de MongoDB Atlas: ")

MONGO_URI = (
    f"mongodb+srv://{DB_USER}:{DB_PASSWORD}"
    "@cluster0.sxjck8h.mongodb.net/?appName=Cluster0"
)

client = MongoClient(MONGO_URI, server_api=ServerApi("1"))

try:
    client.admin.command("ping")
    print("ngrok y MongoDB configurados correctamente")
except Exception as e:
    raise Exception("Ocurrio el siguiente error al conectar: ", e)

db = client["ernesto_investing_ai"]

# Modelos RNN validos para el endpoint /api/rnn (Notebook 4)
MODELOS_RNN = ["LSTM", "BiLSTM", "GRU", "SimpleRNN"]

Password de MongoDB Atlas: ··········
ngrok y MongoDB configurados correctamente


In [19]:
# Paso 3 — Crear la app FastAPI con CORS habilitado + endpoint /api/salud
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Ernesto Investing AI - API")

# CORS habilitado para que el frontend (GitHub Pages) pueda hacer fetch() a esta API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/salud")
def salud():
    """Verifica que el servidor y la base de datos esten operativos, y reporta
    el conteo de documentos de todas las colecciones del proyecto (Notebooks 1-7).
    """
    try:
        client.admin.command("ping")
        return {
            "estado": "operativo",
            "base_datos": "conectada",
            "documentos_precios": db["precios_ohlcv"].count_documents({}),
            "documentos_predicciones": db["predicciones"].count_documents({}),
            "documentos_metricas_modelos": db["metricas_modelos"].count_documents({}),
            "documentos_backtest": db["resultados_backtest"].count_documents({}),
            "documentos_predicciones_lstm": db["predicciones_lstm"].count_documents({}),
            "documentos_metricas_lstm": db["metricas_lstm"].count_documents({}),
            "documentos_sentimiento_nlp": db["sentimiento_nlp"].count_documents({}),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("App FastAPI creada. Endpoint /api/salud listo")

App FastAPI creada. Endpoint /api/salud listo


In [20]:
# Paso 4 — Endpoint de datos de mercado (lee precios_ohlcv de MongoDB, no recalcula)
@app.get("/api/mercado/{ticker}")
def mercado(ticker: str):
    ticker = ticker.upper()

    # Se excluyen _id (no serializable a JSON de forma directa) y created_at (uso interno)
    docs = list(db["precios_ohlcv"].find(
        {"ticker": ticker}, {"_id": 0, "created_at": 0}
    ).sort("fecha", 1))

    if not docs:
        raise HTTPException(status_code=404, detail=f"No hay datos para {ticker}")

    return {"ticker": ticker, "cantidad": len(docs), "datos": docs}

print("Endpoint /api/mercado/{ticker} creado")

Endpoint /api/mercado/{ticker} creado


In [21]:
# Paso 5 — Endpoint de prediccion SVC (lee predicciones y metricas_modelos de MongoDB, Notebook 2)
@app.get("/api/svc/{ticker}")
def svc(ticker: str):
    ticker = ticker.upper()

    prediccion = db["predicciones"].find_one(
        {"ticker": ticker, "modelo": "SVC"}, {"_id": 0, "created_at": 0}
    )
    metricas = db["metricas_modelos"].find_one(
        {"ticker": ticker, "modelo": "SVC"}, {"_id": 0, "fecha_entrenamiento": 0}
    )

    if not prediccion:
        raise HTTPException(status_code=404, detail=f"No hay prediccion SVC para {ticker}")

    return {"ticker": ticker, "prediccion": prediccion, "metricas": metricas}

print("Endpoint /api/svc/{ticker} creado")

Endpoint /api/svc/{ticker} creado


In [22]:
# Paso 6 — Endpoints de las 4 arquitecturas RNN (lee predicciones y metricas_modelos, Notebook 4)
# Comparten las mismas colecciones que el SVC, diferenciadas por el campo "modelo".
@app.get("/api/rnn/{ticker}")
def rnn_todos(ticker: str):
    """Devuelve prediccion y metricas de las 4 arquitecturas recurrentes (LSTM,
    BiLSTM, GRU, SimpleRNN) para un ticker, para poder compararlas en el frontend.
    """
    ticker = ticker.upper()

    resultados = {}
    for modelo in MODELOS_RNN:
        prediccion = db["predicciones"].find_one(
            {"ticker": ticker, "modelo": modelo}, {"_id": 0, "created_at": 0}
        )
        metricas = db["metricas_modelos"].find_one(
            {"ticker": ticker, "modelo": modelo}, {"_id": 0, "fecha_entrenamiento": 0}
        )
        if prediccion:
            resultados[modelo] = {"prediccion": prediccion, "metricas": metricas}

    if not resultados:
        raise HTTPException(status_code=404, detail=f"No hay predicciones RNN para {ticker}")

    return {"ticker": ticker, "modelos": resultados}


@app.get("/api/rnn/{ticker}/{modelo}")
def rnn_modelo(ticker: str, modelo: str):
    """Devuelve prediccion y metricas de UNA arquitectura RNN especifica
    (LSTM, BiLSTM, GRU o SimpleRNN) para un ticker.
    """
    ticker = ticker.upper()

    # Normaliza el nombre del modelo contra la lista conocida (case-insensitive)
    coincidencias = [m for m in MODELOS_RNN if m.lower() == modelo.lower()]
    if not coincidencias:
        raise HTTPException(
            status_code=400,
            detail=f"Modelo '{modelo}' invalido. Usa uno de: {MODELOS_RNN}"
        )
    modelo = coincidencias[0]

    prediccion = db["predicciones"].find_one(
        {"ticker": ticker, "modelo": modelo}, {"_id": 0, "created_at": 0}
    )
    metricas = db["metricas_modelos"].find_one(
        {"ticker": ticker, "modelo": modelo}, {"_id": 0, "fecha_entrenamiento": 0}
    )

    if not prediccion:
        raise HTTPException(
            status_code=404, detail=f"No hay prediccion {modelo} para {ticker}"
        )

    return {"ticker": ticker, "modelo": modelo, "prediccion": prediccion, "metricas": metricas}

print("Endpoints /api/rnn/{ticker} y /api/rnn/{ticker}/{modelo} creados")

Endpoints /api/rnn/{ticker} y /api/rnn/{ticker}/{modelo} creados


In [23]:
# Paso 7 — Endpoint de backtest (lee resultados_backtest de MongoDB, Notebook 5)
@app.get("/api/backtest/{ticker}")
def backtest(ticker: str):
    """Devuelve las metricas, operaciones y curva de equity del backtest de
    cruce de medias moviles (SMA 20/50) para un ticker.
    """
    ticker = ticker.upper()

    resultado = db["resultados_backtest"].find_one(
        {"ticker": ticker, "estrategia": "SMA_20_50"}, {"_id": 0}
    )

    if not resultado:
        raise HTTPException(status_code=404, detail=f"No hay backtest para {ticker}")

    return {"ticker": ticker, "backtest": resultado}

print("Endpoint /api/backtest/{ticker} creado")

Endpoint /api/backtest/{ticker} creado


In [24]:
# Paso 8 — Endpoint del LSTM Regressor (lee predicciones_lstm y metricas_lstm, Notebook 6)
@app.get("/api/lstm-regressor/{ticker}")
def lstm_regressor(ticker: str):
    """Devuelve el pronostico de precios (horizontes 7/14/30/60 dias, con bandas
    de confianza) y las metricas del LSTM Regressor (RMSE, MAE, R2, comparacion
    con ARIMA) para un ticker.
    """
    ticker = ticker.upper()

    prediccion = db["predicciones_lstm"].find_one(
        {"ticker": ticker}, {"_id": 0, "created_at": 0}
    )
    metricas = db["metricas_lstm"].find_one(
        {"ticker": ticker}, {"_id": 0, "fecha_entrenamiento": 0}
    )

    if not prediccion:
        raise HTTPException(
            status_code=404, detail=f"No hay pronostico LSTM Regressor para {ticker}"
        )

    return {"ticker": ticker, "pronostico": prediccion, "metricas": metricas}

print("Endpoint /api/lstm-regressor/{ticker} creado")

Endpoint /api/lstm-regressor/{ticker} creado


In [25]:
# Paso 9 — Endpoint de sentimiento NLP (lee sentimiento_nlp de MongoDB, Notebook 7)
@app.get("/api/sentimiento/{ticker}")
def sentimiento(ticker: str):
    """Devuelve el analisis de sentimiento (VADER) de noticias financieras
    recientes para un ticker: titulares, scores y clasificacion Bullish/Bearish/Neutral.
    """
    ticker = ticker.upper()

    doc = db["sentimiento_nlp"].find_one({"ticker": ticker}, {"_id": 0, "created_at": 0})

    if not doc:
        raise HTTPException(status_code=404, detail=f"No hay sentimiento NLP para {ticker}")

    return {"ticker": ticker, "sentimiento": doc}

print("Endpoint /api/sentimiento/{ticker} creado")

Endpoint /api/sentimiento/{ticker} creado


In [26]:
# Paso 10 — Endpoint de resumen consolidado (combina TODOS los modulos para un ticker)
@app.get("/api/resumen/{ticker}")
def resumen(ticker: str):
    """Endpoint de conveniencia para el dashboard del frontend: en una sola
    llamada trae lo mas relevante de cada modulo (mercado, SVC, RNN, backtest,
    LSTM Regressor y sentimiento) para un ticker. Los modulos sin datos se
    devuelven como null en lugar de fallar toda la respuesta.
    """
    ticker = ticker.upper()

    ultimo_precio = db["precios_ohlcv"].find_one(
        {"ticker": ticker}, {"_id": 0, "created_at": 0}, sort=[("fecha", -1)]
    )

    if not ultimo_precio:
        raise HTTPException(status_code=404, detail=f"No hay datos para {ticker}")

    svc_pred = db["predicciones"].find_one(
        {"ticker": ticker, "modelo": "SVC"}, {"_id": 0, "created_at": 0}
    )

    rnn_preds = {}
    for modelo in MODELOS_RNN:
        p = db["predicciones"].find_one(
            {"ticker": ticker, "modelo": modelo}, {"_id": 0, "created_at": 0}
        )
        if p:
            rnn_preds[modelo] = p

    backtest_doc = db["resultados_backtest"].find_one(
        {"ticker": ticker, "estrategia": "SMA_20_50"},
        {"_id": 0, "operaciones": 0, "curva_equity": 0}  # version ligera: sin el detalle de operaciones
    )

    lstm_reg = db["predicciones_lstm"].find_one(
        {"ticker": ticker}, {"_id": 0, "created_at": 0}
    )

    sentimiento_doc = db["sentimiento_nlp"].find_one(
        {"ticker": ticker}, {"_id": 0, "created_at": 0, "noticias": 0}  # version ligera: sin el detalle de noticias
    )

    return {
        "ticker": ticker,
        "ultimo_precio": ultimo_precio,
        "svc": svc_pred,
        "rnn": rnn_preds if rnn_preds else None,
        "backtest": backtest_doc,
        "lstm_regressor": lstm_reg,
        "sentimiento": sentimiento_doc,
    }

print("Endpoint /api/resumen/{ticker} creado")

Endpoint /api/resumen/{ticker} creado


In [32]:
# Paso 11 — Levantar el servidor uvicorn + exponerlo a Internet con ngrok
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok

nest_asyncio.apply()

# Se cierra cualquier tunel previo para evitar conflictos al re-ejecutar la celda
ngrok.kill()
tunel = ngrok.connect(8000)

print("=" * 65)
print(f"URL PUBLICA DE LA API: {tunel.public_url}")
print("=" * 65)
print(f"Salud:            {tunel.public_url}/api/salud")
print(f"Mercado:          {tunel.public_url}/api/mercado/BVN")
print(f"SVC:              {tunel.public_url}/api/svc/BVN")
print(f"RNN (todas):      {tunel.public_url}/api/rnn/BVN")
print(f"RNN (una):        {tunel.public_url}/api/rnn/BVN/LSTM")
print(f"Backtest:         {tunel.public_url}/api/backtest/BVN")
print(f"LSTM Regressor:   {tunel.public_url}/api/lstm-regressor/BVN")
print(f"Sentimiento NLP:  {tunel.public_url}/api/sentimiento/BVN")
print(f"Resumen:          {tunel.public_url}/api/resumen/BVN")
print(f"Swagger:          {tunel.public_url}/docs")
print("=" * 65)
print()
print(">>> COPIA LA URL PUBLICA <<<")
print(">>> La pegaras en el archivo index.html del frontend <<<")

def correr():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=correr, daemon=True).start()
print("\nAPI operativa. Manten este notebook ejecutandose.")

URL PUBLICA DE LA API: https://acquaint-almost-pretense.ngrok-free.dev
Salud:            https://acquaint-almost-pretense.ngrok-free.dev/api/salud
Mercado:          https://acquaint-almost-pretense.ngrok-free.dev/api/mercado/BVN
SVC:              https://acquaint-almost-pretense.ngrok-free.dev/api/svc/BVN
RNN (todas):      https://acquaint-almost-pretense.ngrok-free.dev/api/rnn/BVN
RNN (una):        https://acquaint-almost-pretense.ngrok-free.dev/api/rnn/BVN/LSTM
Backtest:         https://acquaint-almost-pretense.ngrok-free.dev/api/backtest/BVN
LSTM Regressor:   https://acquaint-almost-pretense.ngrok-free.dev/api/lstm-regressor/BVN
Sentimiento NLP:  https://acquaint-almost-pretense.ngrok-free.dev/api/sentimiento/BVN
Resumen:          https://acquaint-almost-pretense.ngrok-free.dev/api/resumen/BVN
Swagger:          https://acquaint-almost-pretense.ngrok-free.dev/docs

>>> COPIA LA URL PUBLICA <<<
>>> La pegaras en el archivo index.html del frontend <<<

API operativa. Manten este notebo

In [ ]:
# Paso 12 — Celda de verificacion final: probar todos los endpoints desde el propio notebook
import time
import requests

time.sleep(2)  # margen para que uvicorn termine de levantar en el hilo en segundo plano
headers = {"ngrok-skip-browser-warning": "true"}

endpoints = {
    "Salud": f"{tunel.public_url}/api/salud",
    "Mercado (BVN)": f"{tunel.public_url}/api/mercado/BVN",
    "SVC (BVN)": f"{tunel.public_url}/api/svc/BVN",
    "RNN todas (BVN)": f"{tunel.public_url}/api/rnn/BVN",
    "RNN LSTM (BVN)": f"{tunel.public_url}/api/rnn/BVN/LSTM",
    "Backtest (BVN)": f"{tunel.public_url}/api/backtest/BVN",
    "LSTM Regressor (BVN)": f"{tunel.public_url}/api/lstm-regressor/BVN",
    "Sentimiento (BVN)": f"{tunel.public_url}/api/sentimiento/BVN",
    "Resumen (BVN)": f"{tunel.public_url}/api/resumen/BVN",
}

print("Verificando endpoints de la API...")
print()

todos_ok = True
for nombre, url in endpoints.items():
    try:
        r = requests.get(url, headers=headers, timeout=15)
        if r.status_code == 200:
            print(f"  [OK] {nombre}: {r.status_code}")
        else:
            print(f"  [ATENCION] {nombre}: {r.status_code} — {r.text[:100]}")
            todos_ok = False
    except Exception as e:
        print(f"  [ERROR] {nombre}: {e}")
        todos_ok = False

print()
if todos_ok:
    print("Verificacion exitosa: todos los endpoints responden correctamente.")
else:
    print("Atencion: revisar los endpoints marcados arriba antes de conectar el frontend.")
    print("(Si algun modulo aun no fue ejecutado -Notebooks 4/5/6/7-, su endpoint puede devolver 404, no es un error de la API.)")

## Paso 13 — Conectar el frontend

1. **Copia la URL pública** que se mostró en el Paso 11.
2. Abre `index.html` del frontend.
3. Pega la URL en el campo de configuración.
4. Sube el frontend a GitHub Pages.
5. ¡El frontend muestra datos reales de los 7 módulos!

**Nota:** En ngrok gratuito, la primera visita muestra una página de advertencia. Los archivos HTML del frontend incluyen el header `ngrok-skip-browser-warning` para evitarla en las llamadas `fetch()`.

**Nota sobre disponibilidad:** los endpoints `/api/rnn`, `/api/backtest`, `/api/lstm-regressor` y `/api/sentimiento` dependen de que los Notebooks 4, 5, 6 y 7 respectivamente ya hayan sido ejecutados y hayan poblado sus colecciones en MongoDB. Si alguno no fue ejecutado aún, su endpoint devolverá `404` con un mensaje claro en vez de fallar silenciosamente.

**Pipeline completo:** yfinance → MongoDB (`precios_ohlcv`) → SVC / RNN (LSTM, BiLSTM, GRU, SimpleRNN) / Backtest / LSTM Regressor / NLP VADER → MongoDB → **FastAPI + ngrok** → frontend